In [ ]:
with SparkSessionContext(
    app_name="counter-picks"
) as spark:

    #Читаем Stage
    df = spark.table(
        "lakehouse.stage.player_match_heroes"
    )

    #Строим матрицу Match x Hero
    pivot = (
        df.select(
            "match_id",
            "hero_id",
            F.when(F.col("is_radiant"), 1)
             .otherwise(-1)
             .alias("team_indicator")
        )
        .groupBy("match_id")
        .pivot("hero_id")
        .agg(F.first("team_indicator"))
        .fillna(0)
    )

    #Определяем колонки с героями
    feature_cols = sorted(
        [c for c in pivot.columns if c != "match_id"],
        key=int
    )

    print("Количество матчей:", pivot.count())
    print("Количество героев:", len(feature_cols))

    pivot.show(5, truncate=False)

Количество матчей: 309176
Количество героев: 127


[Stage 19:>                                                         (0 + 1) / 1]

+----------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|match_id  |1  |2  |3  |4  |5  |6  |7  |8  |9  |10 |11 |12 |13 |14 |15 |16 |17 |18 |19 |20 |21 |22 |23 |25 |26 |27 |28 |29 |30 |31 |32 |33 |34 |35 |36 |37 |38 |39 |40 |41 |42 |43 |44 |45 |46 |47 |48 |49 |50 |51 |52 |53 |54 |55 |56 |57 |58 |59 |60 |61 |62 |63 |64 |65 |66 |67 |68 |69 |70 |71 |72 |73 |74 |75 |76 |77 |78 |79 |80 |81 |82 |83 |84 |85 |86 |87 |88 |89 |90 |91 |92 |93 |94 |95 |96 |97 |98 |99 |100|101|102|103|104|105|106|107|108|109|110|111|112|113|114|119|120|121|123

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

from pyspark.ml.feature import VectorAssembler, PCA

import numpy as np
from itertools import combinations

In [ ]:
from lakehouse_spark import SparkSessionContext

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, PCA

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType
)

import numpy as np
from itertools import combinations


with SparkSessionContext(
    app_name="counter-picks"
) as spark:

    # 1. ЧТЕНИЕ STAGE

    df = spark.table(
        "lakehouse.stage.player_match_heroes"
    )

    # 2. ПОСТРОЕНИЕ МАТРИЦЫ MATCH x HERO

    pivot = (
        df.select(
            "match_id",
            "hero_id",
            F.when(F.col("is_radiant"), 1)
             .otherwise(-1)
             .alias("team_indicator")
        )
        .groupBy("match_id")
        .pivot("hero_id")
        .agg(F.first("team_indicator"))
        .fillna(0)
    )

    feature_cols = sorted(
        [c for c in pivot.columns if c != "match_id"],
        key=int
    )

    print("Матчей:", pivot.count())
    print("Героев:", len(feature_cols))

    # 3. VECTOR ASSEMBLER

    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features"
    )

    pivot_features = assembler.transform(pivot)

    print("Матрица подготовлена")

    # 4. PCA
    k = min(20, len(feature_cols))

    pca = PCA(
        k=k,
        inputCol="features",
        outputCol="hero_embedding"
    )

    model = pca.fit(pivot_features)

    print("PCA обучена")

    # 5. HERO EMBEDDINGS

    hero_embeddings = model.pc.toArray()

    print("Размерность:", hero_embeddings.shape)

    # 6. COSINE SIMILARITY

    def cosine_sim(v1, v2):

        denominator = (
            np.linalg.norm(v1) *
            np.linalg.norm(v2)
        )

        if denominator == 0:
            return 0.0

        return float(
            np.dot(v1, v2) / denominator
        )

    hero_ids = [int(c) for c in feature_cols]

    similarities = []

    for i, j in combinations(range(len(hero_ids)), 2):

        sim = cosine_sim(
            hero_embeddings[i],
            hero_embeddings[j]
        )

        similarities.append((
            hero_ids[i],
            hero_ids[j],
            sim
        ))

    print("Количество пар:", len(similarities))

    # 7. ИТОГОВАЯ ТАБЛИЦА

    schema = StructType([
        StructField("hero_id_1", IntegerType(), False),
        StructField("hero_id_2", IntegerType(), False),
        StructField("cosine_similarity", DoubleType(), False)
    ])

    similarity_df = spark.createDataFrame(
        similarities,
        schema=schema
    )

    # 8. ДОБАВЛЕНИЕ ИМЁН ГЕРОЕВ

    hero_names = {
        row["hero_id"]: row["hero_name"]
        for row in (
            df.select("hero_id", "hero_name")
            .distinct()
            .collect()
        )
    }

    name_mapping = F.create_map(
        *[
            item
            for hero_id, hero_name in hero_names.items()
            for item in (F.lit(int(hero_id)), F.lit(hero_name))
        ]
    )

    similarity_df = (
        similarity_df
        .withColumn(
            "hero_name_1",
            name_mapping[F.col("hero_id_1")]
        )
        .withColumn(
            "hero_name_2",
            name_mapping[F.col("hero_id_2")]
        )
        .select(
            "hero_id_1",
            "hero_name_1",
            "hero_id_2",
            "hero_name_2",
            "cosine_similarity"
        )
        .orderBy("cosine_similarity")
    )

    # 9. ПРОСМОТР РЕЗУЛЬТАТА

    similarity_df.show(30, truncate=False)

Матчей: 309176
Героев: 127
Матрица подготовлена


PCA обучена
Размерность: (127, 20)
Количество пар: 8001


+---------+-----------------+---------+-----------------+-------------------+
|hero_id_1|hero_name_1      |hero_id_2|hero_name_2      |cosine_similarity  |
+---------+-----------------+---------+-----------------+-------------------+
|36       |Necrophos        |138      |Muerta           |-0.8115658048080112|
|12       |Phantom Lancer   |60       |Night Stalker    |-0.6936397144807652|
|48       |Luna             |60       |Night Stalker    |-0.6906571239153979|
|12       |Phantom Lancer   |49       |Dragon Knight    |-0.671293569626967 |
|74       |Invoker          |120      |Pangolier        |-0.661369255941276 |
|48       |Luna             |97       |Magnus           |-0.6515720353451702|
|4        |Bloodseeker      |60       |Night Stalker    |-0.6458414259351646|
|48       |Luna             |96       |Centaur Warrunner|-0.6338163097476625|
|12       |Phantom Lancer   |97       |Magnus           |-0.6318049834400773|
|12       |Phantom Lancer   |96       |Centaur Warrunner|-0.6182

In [ ]:
import numpy as np

from itertools import combinations

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType
)

from pyspark.ml.feature import VectorAssembler, PCA
from lakehouse_spark import SparkSessionContext


# ШАГИ 3–4. PCA + СТАТИСТИЧЕСКАЯ ПРОВЕРКА КОНТР-ПИКОВ

with SparkSessionContext(app_name="counter-picks") as spark:

    # 1. Загружаем Stage-таблицу

    df = spark.table("lakehouse.stage.player_match_heroes")

    # 2. Строим матрицу match × hero
    #  1 — Radiant
    # -1 — Dire
    #  0 — герой отсутствует в матче

    pivot = (
        df.select(
            "match_id",
            "hero_id",
            F.when(F.col("is_radiant"), 1)
             .otherwise(-1)
             .alias("team_indicator")
        )
        .groupBy("match_id")
        .pivot("hero_id")
        .agg(F.first("team_indicator"))
        .fillna(0)
    )

    feature_cols = sorted(
        [c for c in pivot.columns if c != "match_id"],
        key=int
    )

    # 3. Применяем PCA

    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features"
    )

    pivot_features = assembler.transform(pivot)

    k = min(20, len(feature_cols))

    pca = PCA(
        k=k,
        inputCol="features",
        outputCol="hero_embedding"
    )

    model = pca.fit(pivot_features)

    # Матрица нагрузок PCA:
    # строки — герои, столбцы — главные компоненты

    hero_embeddings = model.pc.toArray()

    #Считаем cosine similarity между героями

    def cosine_sim(v1, v2):
        denominator = np.linalg.norm(v1) * np.linalg.norm(v2)

        if denominator == 0:
            return 0.0

        return float(np.dot(v1, v2) / denominator)

    hero_ids = [int(c) for c in feature_cols]

    similarities = []

    for i, j in combinations(range(len(hero_ids)), 2):

        sim = cosine_sim(
            hero_embeddings[i],
            hero_embeddings[j]
        )

        similarities.append((
            hero_ids[i],
            hero_ids[j],
            sim
        ))

    similarity_schema = StructType([
        StructField("hero_id_1", IntegerType(), False),
        StructField("hero_id_2", IntegerType(), False),
        StructField("cosine_similarity", DoubleType(), False)
    ])

    similarity_raw = spark.createDataFrame(
        similarities,
        schema=similarity_schema
    )

    #Добавляем названия героев через JOIN

    hero_names = (
        df.select("hero_id", "hero_name")
        .dropDuplicates(["hero_id"])
    )

    names_1 = hero_names.select(
        F.col("hero_id").alias("hero_id_1"),
        F.col("hero_name").alias("hero_name_1")
    )

    names_2 = hero_names.select(
        F.col("hero_id").alias("hero_id_2"),
        F.col("hero_name").alias("hero_name_2")
    )

    similarity_df = (
        similarity_raw
        .join(names_1, on="hero_id_1", how="left")
        .join(names_2, on="hero_id_2", how="left")
        .select(
            "hero_id_1",
            "hero_name_1",
            "hero_id_2",
            "hero_name_2",
            "cosine_similarity"
        )
    )

    #СТАТИСТИЧЕСКАЯ ПРОВЕРКА КОНТР-ПИКОВ

    #Оставляем пары с отрицательной similarity

    counter_pairs = (
        similarity_df
        .filter(F.col("cosine_similarity") < 0)
        .select(
            "hero_id_1",
            "hero_name_1",
            "hero_id_2",
            "hero_name_2",
            "cosine_similarity"
        )
    )

    #Формируем пары героев из противоположных команд

    heroes = (
        df.select(
            "match_id",
            "hero_id",
            "hero_name",
            "is_radiant",
            F.col("win").cast("int").alias("win")
        )
        .dropDuplicates(["match_id", "hero_id"])
    )

    enemies = (
        df.select(
            "match_id",
            F.col("hero_id").alias("enemy_hero_id"),
            F.col("hero_name").alias("enemy_hero_name"),
            F.col("is_radiant").alias("enemy_is_radiant")
        )
        .dropDuplicates(["match_id", "enemy_hero_id"])
    )

    matches = (
        heroes
        .join(enemies, on="match_id")
        .filter(
            F.col("is_radiant") != F.col("enemy_is_radiant")
        )
        .select(
            "match_id",
            "hero_id",
            "hero_name",
            "enemy_hero_id",
            "enemy_hero_name",
            "win"
        )
        .dropDuplicates([
            "match_id",
            "hero_id",
            "enemy_hero_id"
        ])
    )

    #Считаем статистику каждой пары

    pair_stats = (
        matches
        .groupBy(
            "hero_id",
            "hero_name",
            "enemy_hero_id",
            "enemy_hero_name"
        )
        .agg(
            F.countDistinct("match_id").alias("matches_count"),
            F.sum("win").alias("hero_wins")
        )
        .withColumn(
            "win_rate",
            F.col("hero_wins") / F.col("matches_count")
        )
    )

    #Рассчитываем базовый Win Rate каждого героя

    baseline = (
        df.select(
            "match_id",
            "hero_id",
            "win"
        )
        .dropDuplicates(["match_id", "hero_id"])
        .groupBy("hero_id")
        .agg(
            F.countDistinct("match_id").alias("total_matches"),
            F.avg(
                F.col("win").cast("double")
            ).alias("baseline_win_rate")
        )
    )

    pair_stats = (
        pair_stats
        .join(baseline, on="hero_id", how="left")
        .withColumn(
            "win_rate_diff",
            F.col("win_rate") - F.col("baseline_win_rate")
        )
    )

    #Рассчитываем 95% доверительный интервал Уилсона


    z = 1.96

    n = F.col("matches_count").cast("double")
    p = F.col("win_rate")

    denominator = 1 + (z ** 2) / n

    center = (
        p + (z ** 2) / (2 * n)
    ) / denominator

    margin = (
        z * F.sqrt(
            (p * (1 - p) / n) +
            ((z ** 2) / (4 * n ** 2))
        )
    ) / denominator

    pair_stats = (
        pair_stats
        .withColumn("ci_lower", center - margin)
        .withColumn("ci_upper", center + margin)
    )

    # 4.6. Преобразуем симметричные пары в направленные
    # A против B
    # B против A

    pairs_forward = counter_pairs.select(
        F.col("hero_id_1").alias("hero_id"),
        F.col("hero_name_1").alias("hero_name"),
        F.col("hero_id_2").alias("enemy_hero_id"),
        F.col("hero_name_2").alias("enemy_hero_name"),
        "cosine_similarity"
    )

    pairs_backward = counter_pairs.select(
        F.col("hero_id_2").alias("hero_id"),
        F.col("hero_name_2").alias("hero_name"),
        F.col("hero_id_1").alias("enemy_hero_id"),
        F.col("hero_name_1").alias("enemy_hero_name"),
        "cosine_similarity"
    )

    directed_pairs = pairs_forward.unionByName(
        pairs_backward
    )

    # 4.7. Объединяем результаты

    final_result = (
        directed_pairs
        .join(
            pair_stats,
            on=[
                "hero_id",
                "hero_name",
                "enemy_hero_id",
                "enemy_hero_name"
            ],
            how="inner"
        )
        .filter(
            F.col("matches_count") >= 30
        )
        .select(
            "hero_id",
            "hero_name",
            "enemy_hero_id",
            "enemy_hero_name",
            "cosine_similarity",
            "matches_count",
            "hero_wins",
            "win_rate",
            "baseline_win_rate",
            "win_rate_diff",
            "ci_lower",
            "ci_upper"
        )
    )

    # 4.8. Получаем результаты и сортируем на Python

    rows = final_result.collect()

    rows_sorted = sorted(
        rows,
        key=lambda row: (
            row["win_rate_diff"]
            if row["win_rate_diff"] is not None
            else float("-inf")
        ),
        reverse=True
    )

    # 4.9. Выводим топ-50

    print(f"Всего найдено направленных пар: {len(rows_sorted)}")

    for row in rows_sorted[:50]:

        print(
            f"{row['hero_name']} против {row['enemy_hero_name']} | "
            f"Similarity: {row['cosine_similarity']:.3f} | "
            f"Matches: {row['matches_count']} | "
            f"WR: {row['win_rate']:.3f} | "
            f"Baseline: {row['baseline_win_rate']:.3f} | "
            f"Difference: {row['win_rate_diff']:.3f} | "
            f"CI: [{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]"
        )

Всего найдено направленных пар: 5286
Chen против Tiny | Similarity: -0.011 | Matches: 30 | WR: 0.633 | Baseline: 0.439 | Difference: 0.194 | CI: [0.455, 0.781]
Night Stalker против Puck | Similarity: -0.217 | Matches: 149 | WR: 0.685 | Baseline: 0.516 | Difference: 0.169 | CI: [0.606, 0.754]
Chen против Dragon Knight | Similarity: -0.230 | Matches: 76 | WR: 0.605 | Baseline: 0.439 | Difference: 0.166 | CI: [0.493, 0.708]
Rubick против Chen | Similarity: -0.265 | Matches: 127 | WR: 0.646 | Baseline: 0.499 | Difference: 0.147 | CI: [0.559, 0.723]
Windranger против Chen | Similarity: -0.210 | Matches: 122 | WR: 0.631 | Baseline: 0.487 | Difference: 0.144 | CI: [0.543, 0.712]
Dawnbreaker против Chen | Similarity: -0.299 | Matches: 41 | WR: 0.659 | Baseline: 0.521 | Difference: 0.137 | CI: [0.505, 0.784]
Lone Druid против Timbersaw | Similarity: -0.124 | Matches: 144 | WR: 0.604 | Baseline: 0.469 | Difference: 0.135 | CI: [0.523, 0.680]
Underlord против Meepo | Similarity: -0.073 | Matches: